# Notebook 16 — Final Project: Full Model Comparison

## Learning Objectives
- Consolidate all models trained in Notebooks 01–15 into one comparison.
- Read saved metrics from all `runs/` subdirectories — no new training.
- Build an architecture comparison table (tasks, inputs, outputs, parameters).
- Build a results comparison table with all available metrics.
- Create a final gallery of key figures.
- Identify which architecture family (CNN, Transformer, U-Net, DETR) excels at which task.

## Big Picture
Throughout this course we have seen Transformers applied to:
- **Text**: classification (NB 06), sequence-to-sequence translation (NB 07)
- **Time series**: forecasting (NB 08)
- **Images**: classification (NB 09–10), segmentation (NB 11–12), detection (NB 13–15)

This notebook reads all saved `*_metrics.json` files, builds summary tables and comparison
visualisations, and updates `runs/SUMMARY.md`.

## Dataset
No new data loading — reads from `runs/` directories.

## Architecture
Summary of all 10 models built in this course:
```
NB  Model                  Task              Input        Output
06  TextTransformer         Text classif.     tokens       1 label
07  Seq2SeqTransformer      Translation       tokens       tokens
08  TimeSeriesTransformer   Forecasting       scalars      scalars
09  SimpleCNN               Image classif.    [1,28,28]    1 label
10  VisionTransformer       Image classif.    [1,28,28]    1 label
11  UNet                    Segmentation      [3,128,128]  [128,128] mask
12  ViTSegmenter            Segmentation      [3,128,128]  [128,128] mask
13  TinySingleObjectDet.    Detection×1       [3,128,128]  1 box + class
14  TinyGridDetector        Detection×N       [3,128,128]  4×4 grid
15  TinyDETR                Detection×N       [3,128,128]  N queries
```

## Theory
No new theory — this notebook is a synthesis and reflection.

**Key patterns across the course:**
1. Transformers = attention + positional encoding + FFN, applied to any token sequence.
2. The 'token' adapts to the domain: word → patch → time step → grid cell → object query.
3. The output head adapts to the task: softmax → pixel logits → box regression.
4. CNNs remain competitive when data is limited; ViTs shine with scale.
5. DETR eliminates engineered components (anchors, NMS) at the cost of training data/epochs.

## Implementation from Scratch
### 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root when running from notebooks/

import json
import csv
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.utils.paths import (
    RUNS_DIR,
    RUNS_SETUP_DIR,
    RUNS_ATTENTION_DIR,
    RUNS_TEXT_CLS_DIR,
    RUNS_TRANSLATION_DIR,
    RUNS_TIMESERIES_DIR,
    RUNS_VIT_CLS_DIR,
    RUNS_SEGMENTATION_DIR,
    RUNS_DETECTION_DIR,
    RUNS_SUMMARY_MD,
)
from src.utils.reporting import save_table_csv

print('Runs directory:', RUNS_DIR)
print('Subdirectories:')
for d in sorted(RUNS_DIR.iterdir()):
    if d.is_dir():
        files = list(d.iterdir())
        print(f'  {d.name:20s}  ({len(files)} files)')

## Dataset
### 2. Load All Saved Metrics

In [ ]:
def load_json_safe(path: Path) -> dict:
    """Load JSON metrics, returning empty dict if file not found."""
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return {}

# Collect all metrics
cnn_m     = load_json_safe(RUNS_VIT_CLS_DIR / 'cnn_metrics.json')
vit_cls_m = load_json_safe(RUNS_VIT_CLS_DIR / 'vit_metrics.json')
unet_m    = load_json_safe(RUNS_SEGMENTATION_DIR / 'unet_metrics.json')
vit_seg_m = load_json_safe(RUNS_SEGMENTATION_DIR / 'vit_segmenter_metrics.json')
single_m  = load_json_safe(RUNS_DETECTION_DIR / 'single_detector_metrics.json')
grid_m    = load_json_safe(RUNS_DETECTION_DIR / 'grid_detector_metrics.json')
detr_m    = load_json_safe(RUNS_DETECTION_DIR / 'tiny_detr_metrics.json')

# Also check earlier notebooks
text_cls_m = load_json_safe(RUNS_TEXT_CLS_DIR / 'metrics.json')
timeseries_m = load_json_safe(RUNS_TIMESERIES_DIR / 'metrics.json')

def fmt(val, key, unit=''):
    if not val or key not in val:
        return 'N/A'
    v = val[key]
    if isinstance(v, float):
        return f'{v:.4f}{unit}'
    return str(v)

print('Metrics loaded successfully.')
print(f'  CNN metrics      : {bool(cnn_m)}')
print(f'  ViT-cls metrics  : {bool(vit_cls_m)}')
print(f'  U-Net metrics    : {bool(unet_m)}')
print(f'  ViT-seg metrics  : {bool(vit_seg_m)}')
print(f'  Single det.      : {bool(single_m)}')
print(f'  Grid det.        : {bool(grid_m)}')
print(f'  TinyDETR         : {bool(detr_m)}')

### 3. Architecture Comparison Table

In [ ]:
ARCH_TABLE = [
    {
        'notebook': 'NB 06',
        'model': 'TextTransformer',
        'task': 'Text Classification',
        'input': 'token IDs',
        'output': '1 label',
        'architecture': 'Embedding + SinPE + TransformerEncoder + MeanPool + Linear',
        'key_component': 'Multi-head self-attention',
        'num_params': text_cls_m.get('num_params', 'N/A'),
    },
    {
        'notebook': 'NB 07',
        'model': 'Seq2SeqTransformer',
        'task': 'Machine Translation',
        'input': 'token IDs (src)',
        'output': 'token IDs (tgt)',
        'architecture': 'Embedding + Encoder + Decoder (cross-attn) + Linear',
        'key_component': 'Cross-attention + causal masking',
        'num_params': 'N/A',
    },
    {
        'notebook': 'NB 08',
        'model': 'TimeSeriesTransformer',
        'task': 'Time Series Forecasting',
        'input': 'scalar series',
        'output': 'forecast scalars',
        'architecture': 'Linear proj + SinPE + TransformerEncoder + MeanPool + MLP',
        'key_component': 'Regression head (MSE)',
        'num_params': timeseries_m.get('num_params', 'N/A'),
    },
    {
        'notebook': 'NB 09',
        'model': 'SimpleCNN',
        'task': 'Image Classification',
        'input': '[1, 28, 28]',
        'output': '10 classes',
        'architecture': 'Conv×2 + BN + MaxPool + Flatten + MLP',
        'key_component': 'Spatial locality + translation equivariance',
        'num_params': cnn_m.get('num_params', 'N/A'),
    },
    {
        'notebook': 'NB 10',
        'model': 'VisionTransformer',
        'task': 'Image Classification',
        'input': '[1, 28, 28]',
        'output': '10 classes',
        'architecture': 'PatchEmb + CLS + LearnedPE + TransformerEncoder + Linear',
        'key_component': 'CLS token + patch self-attention',
        'num_params': vit_cls_m.get('num_params', 'N/A'),
    },
    {
        'notebook': 'NB 11',
        'model': 'UNet',
        'task': 'Semantic Segmentation',
        'input': '[3, 128, 128]',
        'output': '[4, 128, 128] mask',
        'architecture': 'DownBlock×3 + Bottleneck + UpBlock×3 (skip) + Conv1x1',
        'key_component': 'Skip connections for spatial recovery',
        'num_params': unet_m.get('num_params', 'N/A'),
    },
    {
        'notebook': 'NB 12',
        'model': 'ViTSegmenter',
        'task': 'Semantic Segmentation',
        'input': '[3, 128, 128]',
        'output': '[4, 128, 128] mask',
        'architecture': 'PatchEmb + SinPE + TransformerEncoder + ConvTranspose decoder',
        'key_component': 'Token reshape + ConvTranspose upsampling',
        'num_params': vit_seg_m.get('num_params', 'N/A'),
    },
    {
        'notebook': 'NB 13',
        'model': 'TinySingleObjectDetector',
        'task': 'Single-Object Detection',
        'input': '[3, 128, 128]',
        'output': 'class + (cx,cy,w,h)',
        'architecture': 'CNNBackbone + GlobalPool + ClassHead + BoxHead',
        'key_component': 'Shared backbone + dual heads',
        'num_params': single_m.get('num_params', 'N/A'),
    },
    {
        'notebook': 'NB 14',
        'model': 'TinyGridDetector',
        'task': 'Multi-Object Detection',
        'input': '[3, 128, 128]',
        'output': '4×4 grid predictions',
        'architecture': 'CNNBackbone + AdaptivePool(4×4) + Conv1x1(n_pred)',
        'key_component': 'Grid assignment + objectness gating',
        'num_params': grid_m.get('num_params', 'N/A'),
    },
    {
        'notebook': 'NB 15',
        'model': 'TinyDETR',
        'task': 'Multi-Object Detection',
        'input': '[3, 128, 128]',
        'output': 'N query predictions',
        'architecture': 'CNNBackbone + TransformerEncoder + N queries + TransformerDecoder',
        'key_component': 'Object queries + cross-attention (no NMS)',
        'num_params': detr_m.get('num_params', 'N/A'),
    },
]

# Display
print(f'{"NB":<8} {"Model":<30} {"Task":<28} {"Params":>10}')
print('-' * 80)
for r in ARCH_TABLE:
    params = r['num_params']
    params_str = f'{params:,}' if isinstance(params, int) else str(params)
    print(f'{r["notebook"]:<8} {r["model"]:<30} {r["task"]:<28} {params_str:>10}')

In [ ]:
# Save architecture table CSV
save_table_csv(ARCH_TABLE, RUNS_DIR / 'final_comparison_table.csv')
print(f'Architecture table saved to: {RUNS_DIR}/final_comparison_table.csv')

### 4. Results Comparison Table

In [ ]:
RESULTS_TABLE = [
    {
        'model': 'SimpleCNN (NB 09)',
        'task': 'Classification',
        'primary_metric': 'Test Accuracy',
        'value': fmt(cnn_m, 'test_accuracy'),
        'secondary_metric': 'Params',
        'secondary_value': f"{cnn_m.get('num_params', 'N/A'):,}" if isinstance(cnn_m.get('num_params'), int) else 'N/A',
        'notes': '3 epochs, Fashion-MNIST',
    },
    {
        'model': 'ViT (NB 10)',
        'task': 'Classification',
        'primary_metric': 'Test Accuracy',
        'value': fmt(vit_cls_m, 'test_accuracy'),
        'secondary_metric': 'Params',
        'secondary_value': f"{vit_cls_m.get('num_params', 'N/A'):,}" if isinstance(vit_cls_m.get('num_params'), int) else 'N/A',
        'notes': '5 epochs, Fashion-MNIST',
    },
    {
        'model': 'U-Net (NB 11)',
        'task': 'Segmentation',
        'primary_metric': 'Mean IoU',
        'value': fmt(unet_m, 'mean_iou'),
        'secondary_metric': 'Dice',
        'secondary_value': fmt(unet_m, 'dice_score'),
        'notes': '5 epochs, SyntheticShapes',
    },
    {
        'model': 'ViTSegmenter (NB 12)',
        'task': 'Segmentation',
        'primary_metric': 'Mean IoU',
        'value': fmt(vit_seg_m, 'mean_iou'),
        'secondary_metric': 'Dice',
        'secondary_value': fmt(vit_seg_m, 'dice_score'),
        'notes': '5 epochs, SyntheticShapes',
    },
    {
        'model': 'TinySingleObjectDet. (NB 13)',
        'task': 'Detection (×1)',
        'primary_metric': 'Cls Accuracy',
        'value': fmt(single_m, 'cls_accuracy'),
        'secondary_metric': 'Mean IoU',
        'secondary_value': fmt(single_m, 'mean_iou'),
        'notes': '8 epochs, SyntheticShapes',
    },
    {
        'model': 'TinyGridDetector (NB 14)',
        'task': 'Detection (×N)',
        'primary_metric': 'Precision@0.5',
        'value': fmt(grid_m, 'precision_at_iou50'),
        'secondary_metric': 'Recall@0.5',
        'secondary_value': fmt(grid_m, 'recall_at_iou50'),
        'notes': '8 epochs, SyntheticShapes (w/ NMS)',
    },
    {
        'model': 'TinyDETR (NB 15)',
        'task': 'Detection (×N)',
        'primary_metric': 'Precision@0.5',
        'value': fmt(detr_m, 'precision_at_iou50'),
        'secondary_metric': 'Recall@0.5',
        'secondary_value': fmt(detr_m, 'recall_at_iou50'),
        'notes': '10 epochs, SyntheticShapes (no NMS)',
    },
]

print(f'{"Model":<32} {"Task":<18} {"Primary Metric":<18} {"Value":>10} {"Notes"}')
print('-' * 100)
for r in RESULTS_TABLE:
    print(f'{r["model"]:<32} {r["task"]:<18} {r["primary_metric"]:<18} {r["value"]:>10}  {r["notes"]}')

## Sanity Checks

In [ ]:
# Verify expected run directories exist
expected_dirs = [
    RUNS_VIT_CLS_DIR,
    RUNS_SEGMENTATION_DIR,
    RUNS_DETECTION_DIR,
]
for d in expected_dirs:
    status = 'EXISTS' if d.exists() else 'MISSING'
    print(f'  {status:8s} : {d.relative_to(RUNS_DIR.parent)}')

# Count how many metrics files were found
all_metrics = [cnn_m, vit_cls_m, unet_m, vit_seg_m, single_m, grid_m, detr_m]
found = sum(1 for m in all_metrics if m)
print(f'\nMetrics files found: {found}/{len(all_metrics)}')
if found < len(all_metrics):
    print('WARNING: Some metrics files are missing — run earlier notebooks first.')
else:
    print('All metrics files found!')

## Evaluation
### Performance Overview

In [ ]:
# Classification models comparison (if data available)
cls_models = ['SimpleCNN', 'ViT']
cls_accs   = [cnn_m.get('test_accuracy', 0), vit_cls_m.get('test_accuracy', 0)]
cls_params = [cnn_m.get('num_params', 0), vit_cls_m.get('num_params', 0)]
cls_epochs = [cnn_m.get('num_epochs', 0), vit_cls_m.get('num_epochs', 0)]

if any(cls_accs):
    print('Classification models on Fashion-MNIST:')
    for m, a, p, e in zip(cls_models, cls_accs, cls_params, cls_epochs):
        eff = a / (p / 1000) if p else 0
        print(f'  {m:<20}  acc={a:.4f}  params={p:>8,}  epochs={e}  acc/K-params={eff:.4f}')

# Segmentation comparison
seg_models = ['U-Net', 'ViTSegmenter']
seg_ious   = [unet_m.get('mean_iou', 0), vit_seg_m.get('mean_iou', 0)]
seg_dices  = [unet_m.get('dice_score', 0), vit_seg_m.get('dice_score', 0)]

if any(seg_ious):
    print('\nSegmentation models on SyntheticShapes:')
    for m, iou, dice in zip(seg_models, seg_ious, seg_dices):
        print(f'  {m:<20}  mIoU={iou:.4f}  dice={dice:.4f}')

# Detection comparison
det_models  = ['TinySingleObjectDet.', 'TinyGridDetector', 'TinyDETR']
det_metrics = [single_m, grid_m, detr_m]
if any(det_metrics):
    print('\nDetection models on SyntheticShapes:')
    for m, dm in zip(det_models, det_metrics):
        if not dm:
            continue
        if 'cls_accuracy' in dm:
            print(f'  {m:<25}  cls_acc={dm["cls_accuracy"]:.4f}  mIoU={dm.get("mean_iou", 0):.4f}')
        else:
            print(f'  {m:<25}  P@0.5={dm.get("precision_at_iou50", 0):.4f}  R@0.5={dm.get("recall_at_iou50", 0):.4f}')

## Visualization

In [ ]:
# ── Figure 1: Classification comparison bar chart ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy comparison
colors = ['steelblue', 'tomato']
bars = axes[0].bar(cls_models, [a * 100 for a in cls_accs], color=colors,
                   edgecolor='black', linewidth=0.8, width=0.5)
for bar, acc, p in zip(bars, cls_accs, cls_params):
    if acc > 0:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{acc*100:.1f}%\n({p//1000}K params)',
                     ha='center', va='bottom', fontsize=10)
axes[0].set_ylim(0, 110)
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Classification: Fashion-MNIST Test Accuracy')
axes[0].grid(axis='y', alpha=0.3)

# Segmentation comparison
seg_colors = ['cornflowerblue', 'salmon']
x = np.arange(len(seg_models))
w = 0.35
b1 = axes[1].bar(x - w/2, seg_ious,  width=w, color=seg_colors[0], label='Mean IoU',  edgecolor='black', linewidth=0.8)
b2 = axes[1].bar(x + w/2, seg_dices, width=w, color=seg_colors[1], label='Dice Score', edgecolor='black', linewidth=0.8)
for b, v in zip(list(b1) + list(b2), seg_ious + seg_dices):
    if v > 0:
        axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                     f'{v:.3f}', ha='center', va='bottom', fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(seg_models)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('Metric Value')
axes[1].set_title('Segmentation: Mean IoU & Dice')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Model Comparison: Classification & Segmentation', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(RUNS_DIR / 'final_model_comparison.png', dpi=120)
plt.close(fig)
print('Comparison figure saved.')

In [ ]:
# ── Figure 2: Detection model comparison ───────────────────────────────────
det_data = []
for name, dm in zip(det_models, det_metrics):
    if not dm:
        continue
    if 'precision_at_iou50' in dm:
        det_data.append((name, dm['precision_at_iou50'], dm.get('recall_at_iou50', 0)))

if det_data:
    det_names, det_prec, det_rec = zip(*det_data)
    x = np.arange(len(det_names))
    w = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - w/2, det_prec, width=w, color='steelblue', label='Precision@IoU0.5', edgecolor='black', lw=0.8)
    b2 = ax.bar(x + w/2, det_rec,  width=w, color='tomato',    label='Recall@IoU0.5',   edgecolor='black', lw=0.8)
    for b, v in zip(list(b1) + list(b2), list(det_prec) + list(det_rec)):
        if v > 0:
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(' ', '\n') for n in det_names], fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Metric Value')
    ax.set_title('Detection Models: Precision & Recall @IoU=0.5\nSyntheticShapes (multi-object, 3 classes)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(RUNS_DIR / 'final_detection_comparison.png', dpi=120)
    plt.close(fig)
    print('Detection comparison saved.')
else:
    print('Detection metrics not available (run NB 14 and 15 first).')

In [ ]:
# ── Figure 3: Parameter count vs performance scatter ───────────────────────
param_perf_data = []
if cnn_m:     param_perf_data.append(('CNN', cnn_m['num_params'], cnn_m['test_accuracy'] * 100, 'Classification'))
if vit_cls_m: param_perf_data.append(('ViT', vit_cls_m['num_params'], vit_cls_m['test_accuracy'] * 100, 'Classification'))
if unet_m:    param_perf_data.append(('U-Net', unet_m['num_params'], unet_m['mean_iou'] * 100, 'Segmentation (mIoU)'))
if vit_seg_m: param_perf_data.append(('ViTSeg', vit_seg_m['num_params'], vit_seg_m['mean_iou'] * 100, 'Segmentation (mIoU)'))

if param_perf_data:
    fig, ax = plt.subplots(figsize=(8, 5))
    color_map = {'Classification': 'steelblue', 'Segmentation (mIoU)': 'tomato'}
    for name, params, perf, task in param_perf_data:
        ax.scatter(params / 1000, perf, s=120, color=color_map.get(task, 'gray'),
                   edgecolors='black', linewidths=0.8, zorder=3)
        ax.annotate(name, (params / 1000, perf), textcoords='offset points',
                    xytext=(6, 3), fontsize=9)

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, edgecolor='black', label=t)
                       for t, c in color_map.items()]
    ax.legend(handles=legend_elements)
    ax.set_xlabel('Parameters (K)')
    ax.set_ylabel('Performance (accuracy or mIoU ×100)')
    ax.set_title('Parameter Count vs Performance')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(RUNS_DIR / 'final_params_vs_performance.png', dpi=120)
    plt.close(fig)
    print('Params-vs-performance scatter saved.')

In [ ]:
# ── Figure 4: Available saved figures gallery ───────────────────────────────
gallery_paths = [
    (RUNS_VIT_CLS_DIR     / 'cnn_confusion_matrix.png',          'CNN Confusion Matrix'),
    (RUNS_VIT_CLS_DIR     / 'vit_confusion_matrix.png',          'ViT Confusion Matrix'),
    (RUNS_SEGMENTATION_DIR / 'unet_examples.png',                'U-Net Examples'),
    (RUNS_SEGMENTATION_DIR / 'vit_segmenter_examples.png',       'ViT-Seg Examples'),
    (RUNS_DETECTION_DIR   / 'single_detector_predictions.png',   'Single Detector'),
    (RUNS_DETECTION_DIR   / 'grid_detector_predictions.png',     'Grid Detector'),
]

available = [(p, t) for p, t in gallery_paths if p.exists()]
print(f'{len(available)}/{len(gallery_paths)} gallery figures available.')

if available:
    n_cols = 3
    n_rows = (len(available) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
    axes_flat = axes.flatten() if len(available) > 1 else [axes]

    for ax, (path, title) in zip(axes_flat, available):
        img = plt.imread(str(path))
        ax.imshow(img)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.axis('off')

    for ax in axes_flat[len(available):]:
        ax.axis('off')

    plt.suptitle('Course Output Gallery — Selected Figures from All Notebooks', fontsize=13)
    plt.tight_layout()
    fig.savefig(RUNS_DIR / 'final_gallery.png', dpi=80)
    plt.close(fig)
    print('Gallery saved.')
else:
    print('No gallery figures available yet — run earlier notebooks first.')

## Failure Cases

**Cross-notebook failure patterns and lessons:**

1. **CNN vs ViT on small data (NB 09–10)**: ViT needs more data and epochs. On Fashion-MNIST,
   CNN often outperforms ViT trained from scratch with the same budget.

2. **U-Net vs ViT-Segmenter boundaries (NB 11–12)**: Without skip connections, ViT-Segmenter
   produces blurrier boundaries. The 16-pixel patch stride limits spatial precision.

3. **Grid detector cell collision (NB 14)**: Two objects with centres in the same grid cell
   cannot both be detected. DETR's query mechanism solves this.

4. **TinyDETR slow convergence (NB 15)**: With 10 epochs, DETR may not outperform the grid
   detector despite its architectural advantages. Real DETR needs 500 epochs.

5. **Dataset distribution shift**: All models are trained on synthetic geometric shapes with
   fixed colours. Real-world performance would be significantly lower due to texture variety,
   occlusion, lighting changes, and scale variation.

## Exercises

1. **Efficiency frontier**: Plot test accuracy (or mIoU) vs parameter count for all models.
   Draw the Pareto frontier. Which model has the best accuracy-per-parameter trade-off?

2. **Training time comparison**: Add a `training_time_s` field to each metrics JSON (by timing
   the training loop). Plot accuracy vs training time. Which model is most time-efficient?

3. **Transfer learning**: Take the ViT trained on Fashion-MNIST (NB 10) and fine-tune it on
   a new 5-class subset. Compare against training from scratch. How many epochs does
   fine-tuning need vs scratch?

4. **Unified architecture**: Design a single model that can do both classification and
   segmentation (a multi-task ViT). Add a segmentation decoder head alongside the CLS head.
   Train jointly on mixed batches.

5. **Beyond this course**: Read Dosovitskiy et al. (ViT), Carion et al. (DETR), and
   Liu et al. (Swin Transformer). For each paper, identify one architectural difference
   from the toy models in this course and explain the motivation.

## Key Takeaways

**The Transformer recipe** (attention + positional encoding + FFN) adapts to any domain
by changing the 'token' definition:
- Text: word/subword token
- Image: patch token (ViT)
- Time series: time step
- Detection: object query (DETR)
- Segmentation: pixel token (future work)

**Architecture patterns across tasks:**

| Task            | Encoder       | Key component         | Head              |
|-----------------|---------------|-----------------------|-------------------|
| Classification  | CNN or ViT    | Global pooling/CLS    | Linear softmax    |
| Segmentation    | CNN (U-Net)   | Skip connections      | Conv1x1 logits    |
| Segmentation    | ViT           | Token reshape         | ConvTranspose     |
| Detection (×1)  | CNN           | Global pool           | 2 heads (cls+box) |
| Detection (×N)  | CNN+grid      | Objectness + NMS      | Conv1x1 per cell  |
| Detection (×N)  | CNN+ViT       | Object queries        | Set prediction    |

**When to use what:**
- **CNN**: limited data, fast training, spatial locality matters.
- **ViT**: large data, global context, pre-training available.
- **U-Net**: pixel-precise segmentation, medical imaging, skip connections key.
- **DETR**: end-to-end detection without NMS, global reasoning, large data.

## Saved Outputs

In [ ]:
# Update runs/SUMMARY.md
from datetime import datetime

summary_content = (
    '# Runs Summary\n\n'
    'This file is updated automatically after every notebook or script run.\n\n'
    '| Session | Last run | Key Metric | Notes |\n'
    '|---------|----------|------------|-------|\n'
)

now = datetime.now().strftime('%Y-%m-%d %H:%M')
rows = [
    ('cnn_baseline',       cnn_m,    'test_accuracy',       'Fashion-MNIST, 3 epochs'),
    ('vit_classification', vit_cls_m,'test_accuracy',       'Fashion-MNIST, 5 epochs'),
    ('unet_segmentation',  unet_m,   'mean_iou',            'SyntheticShapes, 5 epochs'),
    ('vit_segmentation',   vit_seg_m,'mean_iou',            'SyntheticShapes, 5 epochs'),
    ('single_detection',   single_m, 'cls_accuracy',        'SyntheticShapes, 8 epochs'),
    ('grid_detector',      grid_m,   'precision_at_iou50',  'SyntheticShapes, 8 epochs (NMS)'),
    ('tiny_detr',          detr_m,   'precision_at_iou50',  'SyntheticShapes, 10 epochs (no NMS)'),
]

for session, m, key, notes in rows:
    val = f"{m[key]:.4f}" if m and key in m else 'N/A'
    summary_content += f'| {session} | {now} | {key}={val} | {notes} |\n'

RUNS_SUMMARY_MD.write_text(summary_content)
print(f'SUMMARY.md updated: {RUNS_SUMMARY_MD}')

In [ ]:
# Save final results table CSV
save_table_csv(RESULTS_TABLE, RUNS_DIR / 'final_results_table.csv')
print(f'Results table saved: {RUNS_DIR}/final_results_table.csv')

print('\n=== FINAL OUTPUTS ===')
print(f'  {RUNS_DIR}/final_comparison_table.csv  — architecture overview')
print(f'  {RUNS_DIR}/final_results_table.csv     — performance numbers')
print(f'  {RUNS_DIR}/final_model_comparison.png  — cls + seg comparison chart')
print(f'  {RUNS_DIR}/SUMMARY.md                  — updated session summary')

optional_figs = [
    RUNS_DIR / 'final_detection_comparison.png',
    RUNS_DIR / 'final_params_vs_performance.png',
    RUNS_DIR / 'final_gallery.png',
]
for f in optional_figs:
    status = 'saved' if f.exists() else 'skipped (run earlier notebooks first)'
    print(f'  {f.name:45s} — {status}')

print('\nCourse complete!')